In [9]:
%pip install -q langchain langchain_experimental openai python-dotenv langchain_groq


[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
# os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

os.environ["GROK_API_KEY"] = os.getenv("GROK_API_KEY")


In [11]:
# Load environment variables and initialize the language model
load_dotenv()
# llm = ChatGroq(model="openai/gpt-oss-120b", max_tokens=1000, temperature=0, api_key=os.getenv("GROK_API_KEY"))

# llm = ChatGroq(
#     model="qwen/qwen3-32b",
#     temperature=0,
#     max_tokens=None,
#     reasoning_format="parsed",
#     timeout=None,
#     max_retries=2,
#     api_key=os.getenv("GROK_API_KEY")
#     # other params...
# )
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=os.getenv("GROK_API_KEY")
)

In [12]:
llm.invoke("Heyy")

AIMessage(content='<think>\nOkay, the user said "Heyy". That\'s pretty casual. I should respond in a friendly and welcoming way. Maybe add an emoji to keep it light. Let me check the guidelines. I need to make sure I\'m encouraging them to ask questions or share what\'s on their mind. Keep it open-ended. Alright, something like, "Hey! How can I assist you today? 😊" Yeah, that sounds good. It\'s friendly and invites them to talk more.\n</think>\n\nHey! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 10, 'total_tokens': 124, 'completion_time': 0.232562972, 'completion_tokens_details': None, 'prompt_time': 0.000432301, 'prompt_tokens_details': None, 'queue_time': 0.163350678, 'total_time': 0.232995273}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019df4e4-6f42-7622-8de2-

In [13]:
# Create a simple in-memory store for chat histories
store = {}

def get_chat_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [14]:
# Create the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

In [15]:
# Combine the prompt and model into a runnable chain
chain = prompt | llm

In [16]:
# Wrap the chain with message history
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="input",
    history_messages_key="history"
)

In [17]:
# Example usage
session_id = "user_123"


response1 = chain_with_history.invoke(
    {"input": "Hello! How are you?"},
    config={"configurable": {"session_id": session_id}}
)
print("AI:", response1.content)

response2 = chain_with_history.invoke(
    {"input": "What was my previous message?"},
    config={"configurable": {"session_id": session_id}}
)
print("AI:", response2.content)

AI: <think>
Okay, the user said, "Hello! How are you?" I need to respond appropriately. Since they're asking about my well-being, I should acknowledge their greeting and let them know I'm here to help. I'll keep it friendly and open-ended to invite them to ask for assistance. Let me make sure I don't mention any feelings or emotions since I'm an AI. Something like, "Hi there! I'm just a language model, so I don't have feelings, but I'm here and ready to help you with any questions or tasks you might have. How can I assist you today?" That should work.
</think>

Hi there! I'm just a language model, so I don't have feelings, but I'm here and ready to help you with any questions or tasks you might have. How can I assist you today?
AI: <think>
Okay, the user asked, "What was my previous message?" I need to check the history to see if I can recall their previous message.

Looking back, the conversation started with the user saying "Hello! How are you?" and I responded with a greeting. Then 

In [18]:
# Print the conversation history
print("\nConversation History:")
for message in store[session_id].messages:
    print(f"{message.type}: {message.content}")


Conversation History:
human: Hello! How are you?
ai: <think>
Okay, the user said, "Hello! How are you?" I need to respond appropriately. Since they're asking about my well-being, I should acknowledge their greeting and let them know I'm here to help. I'll keep it friendly and open-ended to invite them to ask for assistance. Let me make sure I don't mention any feelings or emotions since I'm an AI. Something like, "Hi there! I'm just a language model, so I don't have feelings, but I'm here and ready to help you with any questions or tasks you might have. How can I assist you today?" That should work.
</think>

Hi there! I'm just a language model, so I don't have feelings, but I'm here and ready to help you with any questions or tasks you might have. How can I assist you today?
human: What was my previous message?
ai: <think>
Okay, the user asked, "What was my previous message?" I need to check the history to see if I can recall their previous message.

Looking back, the conversation st

In [19]:
llm.invoke("How are you").content

'<think>\nOkay, the user just asked "How are you" in Chinese. I should respond in the same language. I need to be friendly and open about my state as an AI. Let me share that I\'m functioning well and ready to help, while also showing genuine interest in how they\'re doing. Keep it conversational and warm. Make sure there are no markdown formats, just plain text. Also, keep it natural and avoid being too formal. The user might want to continue the conversation, so I should invite them to share more or ask for their needs.\n</think>\n\nI\'m doing well, thank you! I\'m here and ready to chat or help with whatever you need. How are you feeling today? 😊'